# SensIA: Épocas, regularização e estabilidade

Segunda rodada: 16 configurações com 128 neurônios (20/30/40/60 Épocas e alpha 0.03/0.1/0.3/1.0), mais a referência de 32 neurônios, alpha 0.01 e 20 Épocas. **Todas as 17 configurações são repetidas com sementes 42, 123 e 2026: 51 ajustes.** A seleção usa a média do F1 macro na validação; o desvio padrão descreve a variação entre inicializações, não um intervalo de confiança.

O notebook original e seus resultados são preservados. Não avaliamos novamente o teste. A validação jÉ foi utilizada para desenvolvimento: esta rodada não constitui uma estimativa independente de generalizaÉÉo.


In [1]:
from pathlib import Path
from datetime import datetime
import json, hashlib, importlib.metadata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, PredefinedSplit, ParameterGrid
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (accuracy_score, f1_score, log_loss, classification_report,
                             ConfusionMatrixDisplay, roc_curve, roc_auc_score)
from threadpoolctl import threadpool_limits

# Abra a pasta SensIA_VSCode no VS Code antes de executar.
BASE = Path.cwd()
if not (BASE / 'Embeddings').is_dir():
    BASE = BASE / 'SensIA' / 'SensIA_VSCode'
PASTA = BASE / 'Embeddings'
# Se necessário, use um caminho completo, por exemplo:
# PASTA = Path(r'C:\Users\SeuUsuario\Downloads\SensIA_VSCode\Embeddings')
print('Pasta dos embeddings:', PASTA.resolve())
ENTRADA = 'descricao'
SEED = 42
SAIDA = PASTA.parent / 'resultados_refinamento' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
SAIDA.mkdir(parents=True, exist_ok=False)

Pasta dos embeddings: C:\Users\anabe\OneDrive\Documentos\SENSIA_RES\SensIA\SensIA_VSCode\Embeddings


## Integridade
O teste É lido apenas para conferir integridade e sobreposição. Seus rótulos e métricas não participam da busca.


In [2]:
if ENTRADA not in ('descricao', 'transcricao'):
    raise ValueError("ENTRADA deve ser 'descricao' ou 'transcricao'.")
coluna_texto = 'descricao_global' if ENTRADA == 'descricao' else 'transcricao_pronta'
colunas = [f'embedding_{i}' for i in range(1, 1025)]
dados = {}
for split in ('treino', 'validacao', 'teste'):
    arquivo = PASTA / f'embeddings_{split}_{ENTRADA}.csv'
    if not arquivo.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {arquivo}. Confira o caminho e o salvamento dos embeddings.')
    df = pd.read_csv(arquivo, dtype={'video_id': str, 'categoria': str})
    faltam = set(colunas + ['video_id', 'categoria', 'split', coluna_texto]) - set(df.columns)
    if faltam:
        raise ValueError(f'{split}: colunas ausentes: {sorted(faltam)[:8]}')
    if len(df) == 0 or df[['video_id', 'categoria', coluna_texto]].isna().any().any():
        raise ValueError(f'{split}: conjunto vazio ou ID, classe ou texto ausentes.')
    for c in ('video_id', 'categoria', coluna_texto):
        if df[c].str.strip().eq('').any():
            raise ValueError(f'{split}: campo vazio em {c}.')
    if df.video_id.duplicated().any() or not df['split'].eq(split).all():
        raise ValueError(f'{split}: vídeo duplicado ou coluna split inconsistente.')
    vetores = df[colunas].to_numpy(dtype=np.float32)
    if not np.isfinite(vetores).all() or (np.linalg.norm(vetores, axis=1) == 0).any():
        raise ValueError(f'{split}: embeddings inválidos ou vetores nulos.')
    dados[split] = df

classes = np.sort(dados['treino'].categoria.unique())
if len(classes) != 9:
    raise ValueError(f'Esperadas nove classes; encontradas {len(classes)}.')
for split, df in dados.items():
    if set(df.categoria) != set(classes):
        raise ValueError(f'{split}: precisa conter exemplos de todas as nove classes para esta avaliação.')
for a, b in [('treino', 'validacao'), ('treino', 'teste'), ('validacao', 'teste')]:
    ids_comuns = set(dados[a].video_id) & set(dados[b].video_id)
    textos_a = set(dados[a][coluna_texto].str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip())
    textos_b = set(dados[b][coluna_texto].str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip())
    if ids_comuns or textos_a & textos_b:
        raise ValueError(f'Sobreposição entre {a} e {b}. Revise os splits antes de treinar; não exclua linhas silenciosamente.')

print('Integridade básica OK. Embeddings por exemplo:', len(colunas))
display(pd.DataFrame({s: df.categoria.value_counts() for s, df in dados.items()}).loc[classes])

Integridade básica OK. Embeddings por exemplo: 1024
                      treino  validacao  teste
categoria                                     
Art & Creativity         346         78     76
Automotive               362         60     78
Education                352         74     74
Entertainment            346         72     82
Hobbies & Interests      345         79     76
Lifestyle                357         72     71
News & Politics          339         77     84
Science & Technology     357         72     71
Sports                   355         73     72


## Busca com três sementes
Mantemos normalizaÉÉo L2, Adam, taxa 0.001 e batch 64. `n_iter_no_change=61` e `tol=0` permitem os orÉamentos de até 60 Épocas sem parada por estagnação antes deles. Não hÉ early stopping interno.


In [3]:
X_treino = dados['treino'][colunas].to_numpy(dtype=np.float32)
X_validacao = dados['validacao'][colunas].to_numpy(dtype=np.float32)
y_treino = dados['treino'].categoria.to_numpy()
y_validacao = dados['validacao'].categoria.to_numpy()
X_busca = np.concatenate([X_treino, X_validacao])
y_busca = np.concatenate([y_treino, y_validacao])
particao = PredefinedSplit(np.r_[np.full(len(y_treino), -1), np.zeros(len(y_validacao), dtype=int)])
SEMENTES = [42, 123, 2026]
pipeline = Pipeline([
    ('normalizar', Normalizer(norm='l2')),
    ('mlp', MLPClassifier(activation='relu', solver='adam', learning_rate_init=0.001,
        batch_size=64, early_stopping=False, tol=0.0, n_iter_no_change=61,
        shuffle=True))
])
grade = [
    {'mlp__hidden_layer_sizes': [(128,)], 'mlp__alpha': [0.03, 0.1, 0.3, 1.0],
     'mlp__max_iter': [20, 30, 40, 60], 'mlp__random_state': SEMENTES},
    {'mlp__hidden_layer_sizes': [(32,)], 'mlp__alpha': [0.01],
     'mlp__max_iter': [20], 'mlp__random_state': SEMENTES}
]
busca = GridSearchCV(pipeline, grade, scoring='f1_macro', cv=particao,
    refit=False, return_train_score=True, n_jobs=1, verbose=2, error_score='raise')
with warnings.catch_warnings(), threadpool_limits(limits=2):
    warnings.simplefilter('ignore', ConvergenceWarning)
    busca.fit(X_busca, y_busca)
r = pd.DataFrame(busca.cv_results_)
execucoes = pd.DataFrame({
    'neuronios': r.param_mlp__hidden_layer_sizes.map(lambda x: x[0]),
    'alpha': r.param_mlp__alpha.astype(float),
    'epocas': r.param_mlp__max_iter.astype(int),
    'semente': r.param_mlp__random_state.astype(int),
    'f1_treino': r.mean_train_score, 'f1_validacao': r.mean_test_score,
    'segundos': r.mean_fit_time
})
execucoes['gap'] = execucoes.f1_treino - execucoes.f1_validacao
comparacao = execucoes.groupby(['neuronios', 'alpha', 'epocas'], as_index=False).agg(
    f1_validacao_media=('f1_validacao', 'mean'),
    f1_validacao_desvio=('f1_validacao', 'std'),
    f1_validacao_min=('f1_validacao', 'min'),
    f1_validacao_max=('f1_validacao', 'max'),
    f1_treino_media=('f1_treino', 'mean'), gap_medio=('gap', 'mean'),
    repeticoes=('semente', 'count'))
comparacao = comparacao.sort_values(
    ['f1_validacao_media', 'neuronios', 'epocas', 'alpha'],
    ascending=[False, True, True, True]).reset_index(drop=True)
assert comparacao.repeticoes.eq(3).all()
execucoes.to_csv(SAIDA / 'execucoes_por_semente.csv', index=False)
comparacao.to_csv(SAIDA / 'comparacao_medias.csv', index=False)
with pd.option_context('display.max_columns', None, 'display.width', 180):
    display(comparacao.round(4))
print('Selecionamos a configuração pela média, não a semente com maior F1.')


Fitting 1 folds for each of 51 candidates, totalling 51 fits
[CV] END mlp__alpha=0.03, mlp__hidden_layer_sizes=(128,), mlp__max_iter=20, mlp__random_state=42; total time=   3.9s
[CV] END mlp__alpha=0.03, mlp__hidden_layer_sizes=(128,), mlp__max_iter=20, mlp__random_state=123; total time=   4.2s
[CV] END mlp__alpha=0.03, mlp__hidden_layer_sizes=(128,), mlp__max_iter=20, mlp__random_state=2026; total time=   5.3s
[CV] END mlp__alpha=0.03, mlp__hidden_layer_sizes=(128,), mlp__max_iter=30, mlp__random_state=42; total time=   6.7s
[CV] END mlp__alpha=0.03, mlp__hidden_layer_sizes=(128,), mlp__max_iter=30, mlp__random_state=123; total time=   6.1s
[CV] END mlp__alpha=0.03, mlp__hidden_layer_sizes=(128,), mlp__max_iter=30, mlp__random_state=2026; total time=   5.5s
[CV] END mlp__alpha=0.03, mlp__hidden_layer_sizes=(128,), mlp__max_iter=40, mlp__random_state=42; total time=  15.9s
[CV] END mlp__alpha=0.03, mlp__hidden_layer_sizes=(128,), mlp__max_iter=40, mlp__random_state=123; total time=  20

 neuronios  alpha  epocas  f1_validacao_media  f1_validacao_desvio  f1_validacao_min  f1_validacao_max  f1_treino_media  gap_medio  repeticoes
       128   0.10      60              0.6545               0.0052            0.6500            0.6602           0.8057     0.1512           3
       128   0.03      20              0.6510               0.0087            0.6414            0.6585           0.7972     0.1462           3
       128   0.10      20              0.6496               0.0040            0.6452            0.6528           0.7586     0.1090           3
        32   0.01      20              0.6491               0.0035            0.6454            0.6525           0.7560     0.1069           3
       128   0.10      30              0.6460               0.0071            0.6381            0.6518           0.7795     0.1334           3
       128   0.03      30              0.6418               0.0029            0.6388            0.6445           0.8266     0.1848           3

In [4]:
fig, ax = plt.subplots(figsize=(9, 5))
for alpha in [0.03, 0.1, 0.3, 1.0]:
    pontos = comparacao[(comparacao.neuronios == 128) & (comparacao.alpha == alpha)].sort_values('epocas')
    ax.errorbar(pontos.epocas, pontos.f1_validacao_media,
                yerr=pontos.f1_validacao_desvio, marker='o', capsize=4, label=f'alpha={alpha}')
referencia = comparacao[comparacao.neuronios == 32].iloc[0]
ax.axhline(referencia.f1_validacao_media, linestyle='--', color='gray', label='32 neurônios: média')
ax.set(xlabel='Épocas', ylabel='F1 macro de validação', xticks=[20, 30, 40, 60],
       title='Média e desvio padrão entre três sementes')
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(SAIDA / 'epocas_regularizacao.png', dpi=160)
plt.show()


## Modelo selecionado e análise dos erros de validação
Treinamos a configuração escolhida com a semente 42, definida antecipadamente. O relatório e a lista de erros abaixo são de validação e servem para investigar os dados. Uma classe prevista diferente não prova que o rÉtulo original esteja errado. Não altere rótulos apenas para concordar com o modelo.


In [5]:
from sklearn.base import clone
vencedor = comparacao.iloc[0]
parametros = {
    'mlp__hidden_layer_sizes': (int(vencedor.neuronios),),
    'mlp__alpha': float(vencedor.alpha),
    'mlp__max_iter': int(vencedor.epocas),
    'mlp__random_state': SEED
}
modelo = clone(pipeline).set_params(**parametros)
with warnings.catch_warnings(), threadpool_limits(limits=2):
    warnings.simplefilter('ignore', ConvergenceWarning)
    modelo.fit(X_treino, y_treino)
assert modelo.named_steps['mlp'].n_iter_ == int(vencedor.epocas)
prev = modelo.predict(X_validacao)
f1 = f1_score(y_validacao, prev, average='macro')
esperado = execucoes[(execucoes.neuronios == int(vencedor.neuronios)) &
    (execucoes.alpha == float(vencedor.alpha)) &
    (execucoes.epocas == int(vencedor.epocas)) & (execucoes.semente == SEED)].iloc[0]
assert np.isclose(f1, esperado.f1_validacao)
print('Configuração escolhida:', parametros)
print('F1 médio:', round(vencedor.f1_validacao_media, 4),
      '| desvio:', round(vencedor.f1_validacao_desvio, 4))
print('F1 do modelo salvo (semente 42):', round(f1, 4))
relatorio = pd.DataFrame(classification_report(y_validacao, prev, output_dict=True, zero_division=0)).T
display(relatorio.round(4))
relatorio.to_csv(SAIDA / 'relatorio_validacao.csv', encoding='utf-8-sig')
revisao = dados['validacao'][['video_id', 'categoria', coluna_texto]].copy()
revisao['classe_prevista'] = prev
revisao['probabilidade_maxima'] = modelo.predict_proba(X_validacao).max(axis=1)
erros = revisao[revisao.categoria != revisao.classe_prevista].copy()
erros.to_csv(SAIDA / 'erros_validacao.csv', index=False, encoding='utf-8-sig')
confusoes = erros.groupby(['categoria', 'classe_prevista']).size().reset_index(name='quantidade').sort_values('quantidade', ascending=False)
confusoes.to_csv(SAIDA / 'confusoes_validacao.csv', index=False, encoding='utf-8-sig')
display(confusoes.head(15))
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(y_validacao, prev, labels=classes,
    ax=ax, xticks_rotation=65, cmap='Blues', colorbar=False)
fig.tight_layout()
fig.savefig(SAIDA / 'matriz_validacao.png', dpi=160)
plt.show()
joblib.dump(modelo, SAIDA / 'modelo_refinamento.joblib')
config = {'parametros_selecionados': parametros, 'sementes': SEMENTES,
    'criterio': 'Maior média de F1 macro na validação entre três sementes',
    'f1_validacao_media': float(vencedor.f1_validacao_media),
    'f1_validacao_desvio': float(vencedor.f1_validacao_desvio),
    'f1_validacao_modelo_salvo': float(f1), 'teste_avaliado': False,
    'grade': grade, 'parametros_pipeline': modelo.named_steps['mlp'].get_params(),
    'classes': classes.tolist(), 'colunas': colunas,
    'quantidades': {s: len(df) for s, df in dados.items()},
    'sha256_arquivos': {s: hashlib.sha256((PASTA / f'embeddings_{s}_{ENTRADA}.csv').read_bytes()).hexdigest() for s in dados},
    'versoes': {p: importlib.metadata.version(p) for p in ['scikit-learn', 'numpy', 'pandas', 'joblib']}}
(SAIDA / 'configuracao.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print('Resultados salvos em:', SAIDA.resolve())


Configuração escolhida: {'mlp__hidden_layer_sizes': (128,), 'mlp__alpha': 0.1, 'mlp__max_iter': 60, 'mlp__random_state': 42}
F1 médio: 0.6545 | desvio: 0.0052
F1 do modelo salvo (semente 42): 0.6532
                      precision  recall  f1-score   support
Art & Creativity         0.8684  0.8462    0.8571   78.0000
Automotive               0.8382  0.9500    0.8906   60.0000
Education                0.4286  0.3243    0.3692   74.0000
Entertainment            0.5325  0.5694    0.5503   72.0000
Hobbies & Interests      0.5195  0.5063    0.5128   79.0000
Lifestyle                0.6145  0.7083    0.6581   72.0000
News & Politics          0.6627  0.7143    0.6875   77.0000
Science & Technology     0.7273  0.7778    0.7517   72.0000
Sports                   0.6667  0.5479    0.6015   73.0000
accuracy                 0.6545  0.6545    0.6545    0.6545
macro avg                0.6509  0.6605    0.6532  657.0000
weighted avg             0.6475  0.6545    0.6486  657.0000
               catego

## Como interpretar

Se treino sobe e validação cai com mais Épocas, prefira o orÉamento menor. Se o melhor resultado ficar novamente em 60 Épocas, isso justifica investigar a curva alÉm desse ponto, sem garantir melhora. DiferenÉas pequenas diante da variação entre sementes não demonstram superioridade. As três sementes não medem incerteza entre diferentes amostras ou divisões.

Abra `erros_validacao.csv` e `confusoes_validacao.csv` para revisar especialmente Education, Entertainment e Hobbies & Interests. Preserve os splits e documente qualquer revisão independente dos rótulos. Compare futuramente com um classificador linear antes de aumentar a complexidade. Uma avaliação final requer dados novos, reservados depois de concluir o desenvolvimento.

ReferÉncia: [MLPClassifier É scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).
